# Camera Discovery Live Test

Simplified 3-stage pipeline: `TargetResolver → CandidateDiscoveryEngine → ReviewAndValidationPipeline`.

LLMs are used as advisory evidence interpreters/rankers for target intent, geocoder candidate ranking, and candidate semantic review. Deterministic code/tools remain responsible for geometry verification, stream validation, trusted-output authorization, and final artifact writing.

This notebook clones the `main` branch of the GitHub repository by default, installs it in editable mode, runs a live test, and then displays trusted or untrusted camera outputs.

This notebook supports single-location and multi-location queries, for example:

```text
Get me all traffic cameras from California
Get me all cameras from Greenville, Texas
Get me all cameras from London, England and New York, New York
```

The query is intentionally user-controlled. Phrases such as `traffic cameras`, `weather cameras`, or `public live cameras` should be interpreted as camera-type intent, while place names such as `California`, `Greenville, Texas`, or `London, England` are target geography.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil

# Colab/repo bootstrap settings. Override with env vars if needed.
REPO_URL = os.environ.get("CAMERA_DISCOVERY_REPO_URL", "https://github.com/dshipley71/camera-discovery.git")
REPO_BRANCH = os.environ.get("CAMERA_DISCOVERY_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("CAMERA_DISCOVERY_REPO_DIR", "/content/camera-discovery"))

print("Notebook bootstrap")
print("repo url:", REPO_URL)
print("branch:", REPO_BRANCH)
print("repo dir:", REPO_DIR)


In [ ]:
%cd /content

if REPO_DIR.exists():
    print(f"Removing existing repo directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]
print("$", " ".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

src_path = REPO_DIR / "src"
assert (src_path / "camera_discovery").exists(), f"Missing package at {src_path / 'camera_discovery'}"

# Make imports work immediately, even before editable install finishes.
os.environ["PYTHONPATH"] = str(src_path)
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

install_cmd = [sys.executable, "-m", "pip", "install", "-e", ".", "--no-build-isolation"]
print("$", " ".join(install_cmd))
subprocess.run(install_cmd, check=True)


In [ ]:
import camera_discovery
print("camera_discovery import OK:", camera_discovery.__file__)

# Load provider secrets from Colab userdata when available.
# Configure these in Colab as needed:
#   OLLAMA_API_KEY
#   OPENAI_API_KEY
#   OPENAI_BASE_URL
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#   AWS_DEFAULT_REGION
try:
    from google.colab import userdata  # type: ignore
except Exception as exc:
    userdata = None
    print("Colab userdata not available:", repr(exc))

if userdata is not None:
    for key in [
        "OLLAMA_API_KEY",
        "OPENAI_API_KEY",
        "OPENAI_BASE_URL",
        "AWS_ACCESS_KEY_ID",
        "AWS_SECRET_ACCESS_KEY",
        "AWS_SESSION_TOKEN",
        "AWS_DEFAULT_REGION",
    ]:
        if os.environ.get(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
            print(f"Loaded {key} from Colab userdata")

!python -m camera_discovery.cli --help
!python -m camera_discovery.cli run --help


| Profile    | Purpose                    | Behavior                                                                                                                                                                             |
| ---------- | -------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `fast`     | Quick review/discovery run | Validation is minimized/disabled; trusted `camera.geojson` should not be produced unless trust requirements are met; useful for `untrusted_camera_candidates.geojson` review output. |
| `balanced` | Middle-ground run          | More validation than Fast, but avoids the most expensive checks. Good default for routine testing.                                                                                   |
| `full`     | Most thorough run          | Runs the deepest validation path available, intended for trusted output when geometry and stream validation pass.                                                                    |


In [ ]:
RUN_PROFILE = os.environ.get("CAMERA_DISCOVERY_PROFILE", "fast").strip().lower()
if RUN_PROFILE not in {"fast", "balanced", "full"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_PROFILE={RUN_PROFILE!r}; expected fast, balanced, or full")

# User-controlled query. Edit this directly or set CAMERA_DISCOVERY_QUERY in the environment.
# Camera-type terms such as "traffic cameras" are camera intent, not target geography.
USER_QUERY = os.environ.get("CAMERA_DISCOVERY_QUERY", "Get me all traffic cameras from California")

# Other valid examples:
# USER_QUERY = "Get me all cameras from Greenville, Texas"
# USER_QUERY = "Get me all cameras from London, England and New York, New York"

OUTPUT_DIR = Path(os.environ.get("CAMERA_DISCOVERY_OUTPUT_DIR", "runs/notebook-live-test"))
CLEAN_OUTPUT_DIR = os.environ.get("CAMERA_DISCOVERY_CLEAN_OUTPUT_DIR", "true").strip().lower() in {"1", "true", "yes", "on"}

# LLM provider defaults. This application requires a real LLM provider.
# For Ollama Cloud, use model tags that actually exist on the cloud endpoint.
# gemma4:31b-cloud and gemma4:31b-cloud are invalid and cause 404s.
DEFAULT_OLLAMA_CLOUD_MODEL = "gemma4:31b-cloud"

os.environ.setdefault("CAMERA_DISCOVERY_LLM_PROVIDER", "ollama-cloud")
provider = os.environ.get("CAMERA_DISCOVERY_LLM_PROVIDER", "").strip().lower()

def _normalize_model_for_provider(model: str | None, *, provider: str, default_model: str) -> str:
    """Return a provider-compatible model name and repair common bad notebook values."""
    value = (model or "").strip()
    if not value:
        return default_model
    if provider in {"ollama-cloud", "ollama_cloud"}:
        # These tags are common mistakes. Ollama Cloud will return 404 for them.
        invalid_cloud_tags = {
            "gemma4:31b-cloud",
            "gemma4:31b-cloud",
            "gemma3:12b-cloud",
            "gemma3:4b-cloud",
            "gemma3:1b-cloud",
            "gemma3:270m-cloud",
        }
        if value in invalid_cloud_tags:
            print(f"Replacing invalid Ollama Cloud model {value!r} with {default_model!r}")
            return default_model
    return value

# Use setdefault so advanced users can override models before this cell runs.
# Then normalize known-invalid Ollama Cloud model names so the notebook does not fail with 404.
for key in [
    "CAMERA_DISCOVERY_LLM_MODEL",
    "CAMERA_DISCOVERY_TARGET_INTENT_MODEL",
    "CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL",
    "CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL",
]:
    os.environ[key] = _normalize_model_for_provider(
        os.environ.get(key),
        provider=provider,
        default_model=DEFAULT_OLLAMA_CLOUD_MODEL if provider in {"ollama-cloud", "ollama_cloud"} else "gemma3:27b",
    )

DISCOVERY_MODE = os.environ.get("CAMERA_DISCOVERY_DISCOVERY_MODE", "both").strip().lower()
if DISCOVERY_MODE not in {"blind", "directory", "both", "direct"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_DISCOVERY_MODE={DISCOVERY_MODE!r}")

SOURCES_FILE = Path(os.environ.get("CAMERA_DISCOVERY_SOURCES_FILE", "SOURCES.md"))
SEED_URLS = [url.strip() for url in os.environ.get("CAMERA_DISCOVERY_SEED_URLS", "").split(",") if url.strip()]

required_secret_hint = {
    "ollama": "OLLAMA_API_KEY is required only when using Ollama Cloud; local Ollama may not need it.",
    "ollama-cloud": "OLLAMA_API_KEY is required.",
    "ollama_cloud": "OLLAMA_API_KEY is required.",
    "openai-compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "openai_compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "bedrock": "AWS credentials and AWS_DEFAULT_REGION are required.",
}.get(provider, "provider-specific credentials are required")

print("profile:", RUN_PROFILE)
print("query:", USER_QUERY)
print("output:", OUTPUT_DIR)
print("clean output dir before run:", CLEAN_OUTPUT_DIR)
print("provider:", provider)
print("model:", os.environ.get("CAMERA_DISCOVERY_LLM_MODEL"))
print("target intent model:", os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_MODEL"))
print("geocoder referee model:", os.environ.get("CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL"))
print("candidate review model:", os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL"))
print("discovery mode:", DISCOVERY_MODE)
print("sources file:", SOURCES_FILE, "exists=", SOURCES_FILE.exists())
print("seed urls:", len(SEED_URLS))
print("credential hint:", required_secret_hint)

if provider in {"ollama-cloud", "ollama_cloud"} and not os.environ.get("OLLAMA_API_KEY"):
    print("WARNING: OLLAMA_API_KEY is not set. Configure it in Colab userdata or os.environ before running the CLI cell.")

if DISCOVERY_MODE in {"directory", "both"} and not SOURCES_FILE.exists():
    print(f"WARNING: {SOURCES_FILE} does not exist. Directory sources will be empty unless the repo provides it.")
if DISCOVERY_MODE == "direct" and not SEED_URLS:
    raise ValueError("DISCOVERY_MODE=direct requires CAMERA_DISCOVERY_SEED_URLS or --seed-url values")


In [ ]:
cmd = [
    sys.executable, "-m", "camera_discovery.cli", "run", USER_QUERY,
    "--profile", RUN_PROFILE,
    "--output-dir", str(OUTPUT_DIR),
    "--discovery-mode", DISCOVERY_MODE,
    "--sources-file", str(SOURCES_FILE),
]
for url in SEED_URLS:
    cmd.extend(["--seed-url", url])

# Remove stale run artifacts before each live test unless explicitly disabled.
if CLEAN_OUTPUT_DIR and OUTPUT_DIR.exists():
    resolved_output = OUTPUT_DIR.resolve()
    resolved_repo = REPO_DIR.resolve()
    unsafe_roots = {Path("/").resolve(), Path("/content").resolve(), resolved_repo}
    if resolved_output in unsafe_roots:
        raise RuntimeError(f"Refusing to remove unsafe output directory: {resolved_output}")
    print(f"Removing stale output directory: {resolved_output}")
    shutil.rmtree(resolved_output)

print("$", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

run_stdout = OUTPUT_DIR / "notebook_cli_stdout.log"
run_stderr = OUTPUT_DIR / "notebook_cli_stderr.log"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_stdout.write_text(result.stdout or "", encoding="utf-8")
run_stderr.write_text(result.stderr or "", encoding="utf-8")

print(result.stdout)
print(result.stderr)
print("exit:", result.returncode)
print("stdout log:", run_stdout)
print("stderr log:", run_stderr)

if result.returncode != 0:
    raise RuntimeError("camera-discovery run failed; inspect notebook_cli_stdout.log and notebook_cli_stderr.log")


In [ ]:
from pathlib import Path
import json

for rel in ["logs/source_policy_summary.json", "logs/candidate_discovery_summary.json", "logs/run_summary.json"]:
    path = OUTPUT_DIR / rel
    print("---", rel, "exists=", path.exists())
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:4000])
        except Exception as exc:
            print("Could not parse JSON:", repr(exc))
            print(path.read_text(encoding="utf-8")[:1000])


In [ ]:
summary_path = OUTPUT_DIR / "logs" / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
targets = summary.get("targets", [])
print("targets:", len(targets))
for t in targets:
    print(json.dumps({
        "target_id": t.get("target_id"),
        "target_label": t.get("target_label"),
        "canonical_target": t.get("canonical_target"),
        "geometry_status": t.get("geometry_status"),
        "bbox_verified": t.get("bbox_verified"),
        "trust_policy": t.get("trust_policy"),
    }, indent=2))

candidate_summary = summary.get("candidates", {}) or {}
output_summary = summary.get("outputs", {}) or {}
unique_value = candidate_summary.get("unique_count")
if unique_value is None and isinstance(candidate_summary.get("unique"), list):
    unique_value = len(candidate_summary.get("unique"))
coord_value = candidate_summary.get("coordinate_bearing_count")
if coord_value is None and isinstance(candidate_summary.get("coordinate_bearing"), list):
    coord_value = len(candidate_summary.get("coordinate_bearing"))

print(json.dumps({
    "unique_candidates": unique_value,
    "coordinate_bearing": coord_value,
    "trusted_geojson_features": output_summary.get("trusted_geojson_features_written", 0),
    "untrusted_geojson_features": output_summary.get("untrusted_geojson_features_written", 0),
    "trusted_geojson_created": output_summary.get("trusted_geojson_created"),
    "untrusted_geojson_created": output_summary.get("untrusted_geojson_created"),
}, indent=2))


In [ ]:
for rel in [
    'camera.geojson',
    'untrusted_camera_candidates.geojson',
    'map.html',
    'review_artifacts.zip',
    'logs/target_resolution_all.json',
    'logs/target_intent.json',
    'logs/geocoder_referee.json',
    'logs/candidate_semantic_review.json',
    'logs/geocoder_candidate_scores.json',
    'logs/output_summary.json',
]:
    p = OUTPUT_DIR / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

# Per-target diagnostics are written below logs/targets/<target_id>/ and candidates/<target_id>/.
for folder in sorted((OUTPUT_DIR / 'logs' / 'targets').glob('*')) if (OUTPUT_DIR / 'logs' / 'targets').exists() else []:
    print('target diagnostics:', folder.relative_to(OUTPUT_DIR))


## Camera candidate table

This cell loads `camera_candidates_table.csv` first. That CSV is written from all non-rejected review candidates, including candidates that do **not** have latitude/longitude and therefore cannot be mapped. If the CSV is missing, the cell falls back to `camera.geojson` or `untrusted_camera_candidates.geojson`.

In [ ]:
from pathlib import Path
from IPython.display import display

TABLE_PATH = OUTPUT_DIR / "camera_candidates_table.csv"
CAMERA_ROWS = []
GEOJSON_PATH = None

if TABLE_PATH.exists() and TABLE_PATH.stat().st_size > 0:
    print("Selected table:", TABLE_PATH)
    try:
        import pandas as pd
        df = pd.read_csv(TABLE_PATH)
        print("Rows:", len(df))
        display_cols = [
            "name", "target_label", "location_text", "latitude", "longitude",
            "stream_url", "source_url", "thumbnail_url", "media_type", "trust_level",
            "validation_status", "scope_status", "discovery_method", "coordinate_source",
            "review_required",
        ]
        existing_cols = [col for col in display_cols if col in df.columns]
        display(df[existing_cols].head(200))
        CAMERA_ROWS = df.to_dict("records")
        if {"latitude", "longitude"}.issubset(df.columns):
            coordinate_rows = df[df["latitude"].notna() & df["longitude"].notna()]
            print("Coordinate-bearing table rows:", len(coordinate_rows))
        else:
            print("Coordinate-bearing table rows: 0 (latitude/longitude columns missing)")
    except Exception as exc:
        print("Could not display candidate CSV table:", repr(exc))
else:
    print("No camera_candidates_table.csv found; falling back to GeoJSON.")
    from camera_discovery.utils.geojson_viewer import (
        load_camera_rows,
        select_camera_geojson,
        write_camera_table_csv,
    )
    GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
    print("Selected GeoJSON:", GEOJSON_PATH)
    if GEOJSON_PATH is None:
        CAMERA_ROWS = []
        print("No trusted or untrusted camera GeoJSON found yet.")
    else:
        CAMERA_ROWS = load_camera_rows(GEOJSON_PATH)
        table_csv = write_camera_table_csv(OUTPUT_DIR, CAMERA_ROWS)
        print("Rows:", len(CAMERA_ROWS))
        print("CSV table:", table_csv)
        if not CAMERA_ROWS:
            print("GeoJSON exists but contains no camera features.")
        else:
            try:
                import pandas as pd
                df = pd.DataFrame(CAMERA_ROWS)
                display(df.head(200))
            except Exception:
                for row in CAMERA_ROWS[:25]:
                    print(row)


## Interactive camera map

The map below embeds the selected GeoJSON directly into the HTML so it works inside Colab. Click a marker to see camera metadata. If a thumbnail/snapshot URL is present in the GeoJSON properties, the popup shows it. The **Play video** button attempts to play the stream URL with hls.js or native browser video support.

In [ ]:
from IPython.display import HTML, display
from camera_discovery.utils.geojson_viewer import select_camera_geojson, write_embedded_camera_map

MAP_GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
print("Selected GeoJSON for map:", MAP_GEOJSON_PATH)

if MAP_GEOJSON_PATH is None:
    print("No GeoJSON available for map display yet. The table above may still contain non-coordinate candidates.")
else:
    MAP_PATH = write_embedded_camera_map(OUTPUT_DIR, MAP_GEOJSON_PATH, output_name="notebook_camera_map.html")
    print("Notebook map:", MAP_PATH)
    display(HTML(MAP_PATH.read_text(encoding="utf-8")))
